PART 1 — FEATURE ENGINEERING AND EDA

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score

# Load the dataset
df = pd.read_csv("retail_relabel.csv")

# Look at the first few rows
df.head()

,SKU,Category,UnitCost,StorePrice,CostChangePct,Competitor_BigBoxDepot,Competitor_ValueMart,Competitor_QuickShop,WeeklyUnitsSold,DaysSinceLastChange,ElectronicShelfLabel,RelabelCost,Relabel
0,SKU10000,Grocery,1.74,3.06,6.98,2.46,3.34,2.22,71,108,0,0.47,1
1,SKU10001,Electronics,31.67,54.91,5.86,42.03,52.81,40.01,34,51,0,0.45,0
2,SKU10002,Household,17.18,24.37,2.27,26.55,28.43,26.72,91,196,1,0.06,1
3,SKU10003,Household,12.08,15.76,-8.49,15.80,15.93,18.13,54,254,1,0.05,1
4,SKU10004,Household,4.75,6.97,7.41,6.57,6.96,6.41,52,121,0,1.75,0


In [2]:
# Create competitor average
df["CompAvg"] = df[
    ["Competitor_BigBoxDepot",
     "Competitor_ValueMart",
     "Competitor_QuickShop"]
].mean(axis=1)

# Create price gap variables
df["GapPct"] = 100 * (df["StorePrice"] - df["CompAvg"]) / df["CompAvg"]
df["AbsGapPct"] = abs(df["GapPct"])

# Create log units
df["LogUnits"] = np.log(df["WeeklyUnitsSold"])

# Create margin percentage
df["MarginPct"] = 100 * (df["StorePrice"] - df["UnitCost"]) / df["StorePrice"]

# Overall relabel rate
print("Overall relabel rate:")
print(df["Relabel"].mean())

# Relabel rate by category
print("\nRelabel rate by Category:")
print(df.groupby("Category")["Relabel"].mean())

# Relabel rate by ESL
print("\nRelabel rate by Electronic Shelf Label:")
print(df.groupby("ElectronicShelfLabel")["Relabel"].mean())

Overall relabel rate:
0.4

Relabel rate by Category:
Category
Apparel         0.390244
Electronics     0.439791
Grocery         0.411523
Household       0.378378
PersonalCare    0.383333
Name: Relabel, dtype: float64

Relabel rate by Electronic Shelf Label:
ElectronicShelfLabel
0    0.338756
1    0.540659
Name: Relabel, dtype: float64


ESL items had a higher relabeling rate (54.07%) than non-ESL items (33.88%). This is consistent with the menu-cost hypothesis: because ESL makes price changes much cheaper and easier, the store can reprice items more frequently. Paper tags have higher labor and material costs, creating a greater deterrent to relabeling.


PART 2 — A NAIVE MODEL

In [3]:
import statsmodels.formula.api as smf

# Model 1
model1 = smf.logit(
    "Relabel ~ GapPct + UnitCost + RelabelCost",
    data=df
).fit()


print("\nMODEL 1 RESULTS")
print("====================")

print(model1.summary())

# Model statistics
print("\nPseudo R-squared:", model1.prsquared)
print("AIC:", model1.aic)

Optimization terminated successfully.
         Current function value: 0.621007
         Iterations 5

MODEL 1 RESULTS
                           Logit Regression Results                           
Dep. Variable:                Relabel   No. Observations:                 1500
Model:                          Logit   Df Residuals:                     1496
Method:                           MLE   Df Model:                            3
Date:                Wed, 30 Sep 2026   Pseudo R-squ.:                 0.07727
Time:                        13:00:54   Log-Likelihood:                -931.51
converged:                       True   LL-Null:                       -1009.5
Covariance Type:            nonrobust   LLR p-value:                 1.328e-33
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       0.0589      0.098      0.603      0.547      -0.133       0.250
GapPct   

The signed gap is the wrong functional form because both directions of mispricing can lead to a price change. An item **10% below competitors** may need a price increase, while an item **10% above competitors** may need a price decrease. Both represent a **10% misalignment**, so the model should use **AbsGapPct** rather than the signed GapPct.


### Part 3 – Economically Motivated Model
#### 5. Model 2: Absolute Gap, Cost Pressure, Velocity, Menu Cost, Staleness

In [4]:
model2 =smf.logit("Relabel ~ AbsGapPct + CostChangePct + LogUnits + RelabelCost"
               " + DaysSinceLastChange + UnitCost", data=df).fit()
print(model2.summary())

Optimization terminated successfully.
         Current function value: 0.487939
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                Relabel   No. Observations:                 1500
Model:                          Logit   Df Residuals:                     1493
Method:                           MLE   Df Model:                            6
Date:                Wed, 30 Sep 2026   Pseudo R-squ.:                  0.2750
Time:                        13:00:54   Log-Likelihood:                -731.91
converged:                       True   LL-Null:                       -1009.5
Covariance Type:            nonrobust   LLR p-value:                1.059e-116
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -4.2772      0.567     -7.540      0.000      -5.389      -3.165
Ab

#### 6. Odds Ratios with 95% Confidence Intervals

In [5]:
conf = model2.conf_int()
or_table = pd.DataFrame({"OddsRatio": np.exp(model2.params),
                         "CI_low": np.exp(conf[0]),
                         "CI_high": np.exp(conf[1]),
                         "p_value": model2.pvalues})
print(or_table.round(4))

                     OddsRatio  CI_low  CI_high  p_value
Intercept               0.0139  0.0046   0.0422    0.000
AbsGapPct               1.1946  1.1692   1.2205    0.000
CostChangePct           1.1361  1.0992   1.1742    0.000
LogUnits                1.5410  1.1919   1.9923    0.001
RelabelCost             0.4236  0.3585   0.5006    0.000
DaysSinceLastChange     1.0040  1.0028   1.0053    0.000
UnitCost                1.0004  0.9931   1.0077    0.919


#### 8. Model Comparison: Pseudo R², AIC, BIC

In [6]:
compare = pd.DataFrame({"Model 1": [model1.prsquared, model1.aic, model1.bic],
                        "Model 2": [model2.prsquared, model2.aic, model2.bic]},
                       index=["Pseudo R2", "AIC", "BIC"])
print(compare.round(3))

            Model 1   Model 2
Pseudo R2     0.077     0.275
AIC        1871.020  1477.817
BIC        1892.273  1515.009


### Part 4 – Predictive Evaluation
#### 70/30 Stratified Split (random_state = 7), Confusion Matrix, Accuracy, AUC


In [7]:
train, test = train_test_split(df, test_size=0.3, stratify=df["Relabel"], random_state=7)
f2 = "Relabel ~ AbsGapPct + CostChangePct + LogUnits + RelabelCost + DaysSinceLastChange + UnitCost"
model3 = smf.logit(f2, data=train).fit()

p_test = model3.predict(test)
y_pred = (p_test >= 0.5).astype(int)
print(confusion_matrix(test["Relabel"], y_pred))
print("Accuracy:", accuracy_score(test["Relabel"], y_pred))
print("AUC:", roc_auc_score(test["Relabel"], p_test))

Optimization terminated successfully.
         Current function value: 0.501281
         Iterations 6
[[233  37]
 [ 64 116]]
Accuracy: 0.7755555555555556
AUC: 0.8533127572016461


Part 5

In [8]:
df['ExpectedWeeklyLoss'] = (
    df['AbsGapPct']/100 *
    df['StorePrice'] *
    df['WeeklyUnitsSold']
)

df['NetBenefit'] = (
    df['ExpectedWeeklyLoss']
    - df['RelabelCost']
)

In [9]:
df.sort_values(
    'NetBenefit',
    ascending=False
)[['SKU','NetBenefit']].head(10)

,SKU,NetBenefit
117,SKU10117,5292.816094
1432,SKU11432,3947.799598
583,SKU10583,2746.732571
70,SKU10070,2089.142848
1203,SKU11203,2016.527453
1212,SKU11212,1924.199589
1085,SKU11085,1898.507493
1325,SKU11325,1643.968641
728,SKU10728,1613.694309
860,SKU10860,1434.688541


Approximately 98.8% of SKUs would recoup the relabel cost within a single week of corrected pricing.